# 📦 Supplier Evaluation Environment Setup and Scoring Notebook
This notebook automates environment setup and performs supplier grading using both offline Word document and live URL data.

In [18]:
import docx
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 1️⃣ Load Supplier Word Document
def read_supplier_doc(path):
    doc = docx.Document(path)
    supplier_data = {}
    current_supplier = None
    for para in doc.paragraphs:
        if para.style.name.startswith('Heading'):
            current_supplier = para.text.strip()
            supplier_data[current_supplier] = ""
        elif current_supplier:
            supplier_data[current_supplier] += para.text.strip() + " "
    return supplier_data

word_path = r"C:\Users\srahman3\OneDrive - The University of Texas at El Paso\Desktop\Spring 25\Resarrch\scmDis\Supplier_Grading_Detailed_Descriptions.docx"
offline_data = read_supplier_doc(word_path)

# 2️⃣ Crawl Supplier URLs
supplier_urls = {
    "Supplier 1": "https://www.3m.com/3M/en_US/company-us/about-3m/",
    "Supplier 2": "https://en.wikipedia.org/wiki/Foxconn",
    "Supplier 3": "https://www.texhong.com/en/development",
    "Supplier 4": "https://en.wikipedia.org/wiki/Yue_Yuen_Industrial",
    "Supplier 5": "https://en.pouchen.com/about_pouchen.php",
    "Supplier 6": "https://www.tsmc.com/english/aboutTSMC",
    "Supplier 7": ""
}

def crawl_supplier_website(url):
    if not url:
        return ""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        return ' '.join([p.get_text() for p in soup.find_all('p')])
    except:
        return ""

online_data = {supplier: crawl_supplier_website(url) for supplier, url in supplier_urls.items()}

# 3️⃣ Combine Data
combined_data = {}
for supplier in supplier_urls.keys():
    combined_data[supplier] = offline_data.get(supplier, "") + " " + online_data.get(supplier, "")

criteria_keywords = [
    'cost', 'lead', 'quality', 'security', 'compliance', 'sustainability', 
    'inspection', 'risk', 'traceability', 'environment', 'documentation', 
    'cybersecurity', 'FIFO', 'innovation'
]

# 4️⃣ TF-IDF Scoring
all_texts = list(combined_data.values())
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_texts)
criteria_text = " ".join(criteria_keywords)
criteria_vector = vectorizer.transform([criteria_text])

scores = cosine_similarity(tfidf_matrix, criteria_vector).flatten() * 100  # Scale to percentage

# 5️⃣ Final Results
results = {supplier: round(score, 2) for supplier, score in zip(combined_data.keys(), scores)}
df_results = pd.DataFrame(list(results.items()), columns=["Supplier", "Score (%)"])
print(df_results)

# Optional: Save results
df_results.to_csv("Final_Supplier_Grading_Scores_TFIDF.csv", index=False)


     Supplier  Score (%)
0  Supplier 1      11.79
1  Supplier 2       0.41
2  Supplier 3      18.31
3  Supplier 4      10.01
4  Supplier 5      16.05
5  Supplier 6       2.10
6  Supplier 7      21.23


In [20]:
# ✅ Final Code with Individual Criteria Scoring and Total Score Calculation using TF-IDF

import docx
import requests
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 1️⃣ Load Supplier Word Document
def read_supplier_doc(path):
    doc = docx.Document(path)
    supplier_data = {}
    current_supplier = None
    for para in doc.paragraphs:
        if para.style.name.startswith('Heading'):
            current_supplier = para.text.strip()
            supplier_data[current_supplier] = ""
        elif current_supplier:
            supplier_data[current_supplier] += para.text.strip() + " "
    return supplier_data

# Example Path (Update this accordingly)
word_path = r"C:\Users\srahman3\OneDrive - The University of Texas at El Paso\Desktop\Spring 25\Resarrch\scmDis\Supplier_Grading_Detailed_Descriptions.docx"
offline_data = read_supplier_doc(word_path)

# 2️⃣ Crawl Supplier URLs
supplier_urls = {
    "Supplier 1": "https://www.3m.com/3M/en_US/company-us/about-3m/",
    "Supplier 2": "https://en.wikipedia.org/wiki/Foxconn",
    "Supplier 3": "https://www.texhong.com/en/development",
    "Supplier 4": "https://en.wikipedia.org/wiki/Yue_Yuen_Industrial",
    "Supplier 5": "https://en.pouchen.com/about_pouchen.php",
    "Supplier 6": "https://www.tsmc.com/english/aboutTSMC",
    "Supplier 7": ""
}

def crawl_supplier_website(url):
    if not url:
        return ""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        return ' '.join([p.get_text() for p in soup.find_all('p')])
    except:
        return ""

online_data = {supplier: crawl_supplier_website(url) for supplier, url in supplier_urls.items()}

# 3️⃣ Combine Data
combined_data = {}
for supplier in supplier_urls.keys():
    combined_data[supplier] = offline_data.get(supplier, "") + " " + online_data.get(supplier, "")

criteria_keywords = [
    'cost', 'lead', 'quality', 'security', 'compliance', 'sustainability', 
    'inspection', 'risk', 'traceability', 'environment', 'documentation', 
    'cybersecurity', 'FIFO', 'innovation'
]

# 4️⃣ TF-IDF Scoring with Individual Criteria
all_texts = list(combined_data.values())
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_texts)

individual_scores = {}
for crit in criteria_keywords:
    crit_vector = vectorizer.transform([crit])
    scores = cosine_similarity(tfidf_matrix, crit_vector).flatten() * 100  # Scale to percentage
    individual_scores[crit] = scores

# Combine into DataFrame
supplier_names = list(combined_data.keys())
df_individual_scores = pd.DataFrame(individual_scores, index=supplier_names)
df_individual_scores["Total Score (%)"] = df_individual_scores.mean(axis=1).round(2)

# 5️⃣ Final Output
print(df_individual_scores)

# Optional: Save Final Scoring Table
df_individual_scores.to_csv("Final_Supplier_Scoring_Detailed_TFIDF.csv")

# ✅ Done! This will show individual criteria scores and total aggregated score.


                cost  lead    quality  security  compliance  sustainability  \
Supplier 1  7.123693   0.0   4.207368  0.000000    3.561846        0.000000   
Supplier 2  0.000000   0.0   0.170743  0.752979    0.000000        0.501986   
Supplier 3  0.000000   0.0  11.516501  5.643100    6.499711        0.000000   
Supplier 4  6.767502   0.0   9.992489  2.937799    0.000000        5.875598   
Supplier 5  6.935823   0.0  12.289226  0.000000    0.000000       12.043471   
Supplier 6  0.000000   0.0  12.176795  0.000000    0.000000        0.000000   
Supplier 7  0.000000   0.0   8.513370  6.257343   14.414389        6.257343   

            inspection       risk  traceability  environment  documentation  \
Supplier 1    9.277268   0.000000      3.561846          0.0       4.167038   
Supplier 2    0.250993   0.000000      0.000000          0.0       0.000000   
Supplier 3    0.000000  22.812223      0.000000          0.0       0.000000   
Supplier 4    2.937799   0.000000      3.383751    

In [3]:
# ✅ Final Integrated ANN Pipeline for Supplier Scoring
from tensorflow.keras.models import load_model
import docx
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import mean_squared_error, r2_score

# 1️⃣ Load Supplier Word Document
def read_supplier_doc(path):
    doc = docx.Document(path)
    supplier_data = {}
    current_supplier = None
    for para in doc.paragraphs:
        if para.style.name.startswith('Heading'):
            current_supplier = para.text.strip()
            supplier_data[current_supplier] = ""
        elif current_supplier:
            supplier_data[current_supplier] += para.text.strip() + " "
    return supplier_data

# Example path (Update accordingly)
word_path = r"C:\Users\srahman3\OneDrive - The University of Texas at El Paso\Desktop\Spring 25\Resarrch\scmDis\Supplier_Grading_Detailed_Descriptions.docx"
offline_data = read_supplier_doc(word_path)

# 2️⃣ Crawl Supplier URLs
supplier_urls = {
    "Supplier 1": "https://www.3m.com/3M/en_US/company-us/about-3m/",
    "Supplier 2": "https://en.wikipedia.org/wiki/Foxconn",
    "Supplier 3": "https://www.texhong.com/en/development",
    "Supplier 4": "https://en.wikipedia.org/wiki/Yue_Yuen_Industrial",
    "Supplier 5": "https://en.pouchen.com/about_pouchen.php",
    "Supplier 6": "https://www.tsmc.com/english/aboutTSMC",
    "Supplier 7": ""
}

def crawl_supplier_website(url):
    if not url:
        return ""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        return ' '.join([p.get_text() for p in soup.find_all('p')])
    except:
        return ""

online_data = {supplier: crawl_supplier_website(url) for supplier, url in supplier_urls.items()}

# 3️⃣ Combine Data
combined_data = {}
for supplier in supplier_urls.keys():
    combined_data[supplier] = offline_data.get(supplier, "") + " " + online_data.get(supplier, "")

# 4️⃣ TF-IDF Vectorization
texts = list(combined_data.values())
suppliers = list(combined_data.keys())
vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(texts).toarray()

# Simulate or Load Actual Scores (replace this with real scores when available)
np.random.seed(42)
y = np.random.uniform(60, 95, size=len(X))

# 5️⃣ ANN Model for Prediction
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Sequential()
model.add(Dense(256, input_dim=X.shape[1], activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dense(1))  # Output layer for regression

model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
model.fit(X_train, y_train, epochs=100, batch_size=8, verbose=0, validation_split=0.1)

# 6️⃣ Evaluate Model
predictions = model.predict(X_test).flatten()
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

evaluation_results = f"✅ ANN Model Evaluation:\n- MSE: {mse:.2f}\n- R² Score: {r2:.2f}"

# 7️⃣ Predict New Supplier Example
def predict_supplier_score(new_text_list):
    new_vector = vectorizer.transform(new_text_list).toarray()
    predicted_score = model.predict(new_vector).flatten()[0]
    return round(predicted_score, 2)

new_supplier_text = ["High quality standards, excellent sustainability efforts, strong cybersecurity policies."]
predicted_new_score = predict_supplier_score(new_supplier_text)

# 8️⃣ Output Results
supplier_predictions = {supplier: round(score, 2) for supplier, score in zip(suppliers, model.predict(X).flatten())}
df_predictions = pd.DataFrame(list(supplier_predictions.items()), columns=["Supplier", "Predicted Score (%)"])

# Save the final results
final_csv_path = "Final_Supplier_Scoring_ANN.csv"
df_predictions.to_csv(final_csv_path, index=False)

evaluation_results, predicted_new_score, df_predictions.head()
model_save_path = "supplier_ann_model.h5"
model.save(model_save_path)

print(f"Model saved at: {model_save_path}")


1/1 [==============================] - 0s 21ms/step
Model saved at: supplier_ann_model.h5


In [4]:
from tensorflow.keras.models import load_model

# Load the saved model
model = load_model("supplier_ann_model.h5")  # Ensure this file exists in your working directory

# Define Prediction Function
def predict_supplier_score_from_text(text, vectorizer_model, model):
    text_vector = vectorizer_model.transform([text]).toarray()
    predicted_score = model.predict(text_vector).flatten()[0]
    return round(predicted_score, 2)

# Supplier Text Example
supplier_text = """
Supplier 2 stands out for its strong commitment to sustainability and environmental responsibility. 
This supplier has successfully integrated advanced 5S practices across its production and storage areas, 
creating an organized, clean, and efficient workspace. Employees receive regular training on sustainability initiatives, 
and the supplier actively monitors and reduces its carbon footprint. Although Supplier 2 excels in environmental and 
workplace organization practices, there is room for improvement in their incoming inspection control procedures. 
Supplier 3 Their statistical process control tools are underutilized, leading to occasional lapses in early detection of process variations. 
However, the supplier compensates for this by maintaining high standards in production process control, ensuring final product quality. 
Supplier 2’s dedication to sustainable production and its alignment with global ESG standards make it a preferred partner for companies focusing on green procurement.  
Supplier 4The supplier is also actively pursuing ISO 14001 certification to further solidify its commitment to environmental excellence. 
Regular audits and transparent reporting help this supplier build trust and demonstrate accountability to its customers.
"""

# Predict the Supplier Score
predicted_score = predict_supplier_score_from_text(supplier_text, vectorizer, model)
print(f"📢 Predicted Supplier 3 Score: {predicted_score}%")


1/1 [==============================] - 0s 39ms/step
📢 Predicted Supplier 3 Score: 57.25%


In [18]:
from tensorflow.keras.models import load_model
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1️⃣ Load the Saved ANN Model
model = load_model("supplier_ann_model.h5")

# 2️⃣ Supplier Data and URLs
supplier_texts = {
    "Supplier 1": "High quality standards, good sustainability, moderate cybersecurity measures.",
    "Supplier 2": """Supplier 2 stands out for its strong commitment to sustainability and environmental responsibility. 
    This supplier has successfully integrated advanced 5S practices across its production and storage areas, 
    creating an organized, clean, and efficient workspace. Employees receive regular training on sustainability initiatives, 
    and the supplier actively monitors and reduces its carbon footprint. Although Supplier 2 excels in environmental and 
    workplace organization practices, there is room for improvement in their incoming inspection control procedures. 
    Supplier 3 Their statistical process control tools are underutilized, leading to occasional lapses in early detection of process variations. 
    However, the supplier compensates for this by maintaining high standards in production process control, ensuring final product quality. 
    Supplier 2’s dedication to sustainable production and its alignment with global ESG standards make it a preferred partner for companies focusing on green procurement.  
    Supplier 4The supplier is also actively pursuing ISO 14001 certification to further solidify its commitment to environmental excellence. 
    Regular audits and transparent reporting help this supplier build trust and demonstrate accountability to its customers.""",
    "Supplier 3": "Strong in security protocols, excellent compliance, innovation-driven approach.",
}

supplier_urls = {
    "Supplier 1": "https://en.wikipedia.org/wiki/Yue_Yuen_Industrial",
    "Supplier 2": "https://www.tsmc.com/english/aboutTSMC",
    "Supplier 3": "https://en.pouchen.com/about_pouchen.php",
}

criteria_keywords = ['cost', 'lead', 'quality', 'efficient workspace']

# 3️⃣ Web Crawling Function
def crawl_supplier_website(url):
    if not url:
        return ""
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')
        return ' '.join([p.get_text() for p in soup.find_all('p')])
    except Exception as e:
        print(f"Failed to crawl {url}: {e}")
        return ""

# 4️⃣ Merge Crawled Data with Existing Text
for supplier, url in supplier_urls.items():
    crawled_content = crawl_supplier_website(url)
    if supplier in supplier_texts:
        supplier_texts[supplier] += " " + crawled_content
    else:
        supplier_texts[supplier] = crawled_content

# 5️⃣ Prepare TF-IDF Vectorizer from Combined Data
texts = list(supplier_texts.values())
vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(texts).toarray()

# 6️⃣ Generate Predictions Using Loaded ANN Model
supplier_names = list(supplier_texts.keys())
predicted_scores = model.predict(X).flatten()
predicted_scores = [round(score, 2) for score in predicted_scores]

# 7️⃣ Calculate Individual Criteria Scores (Cosine Similarity)
individual_criteria_scores = {}
for i, supplier in enumerate(supplier_names):
    text_vector = X[i].reshape(1, -1)
    criteria_scores = {}
    for crit in criteria_keywords:
        crit_vector = vectorizer.transform([crit]).toarray()
        sim = cosine_similarity(text_vector, crit_vector).flatten()[0] * 100  # Scale to percentage
        criteria_scores[crit] = round(sim, 2)
    individual_criteria_scores[supplier] = criteria_scores

# 8️⃣ Combine All Results into a Final DataFrame
df_scores = pd.DataFrame.from_dict(individual_criteria_scores, orient='index')
df_scores["Predicted Total Score (%)"] = predicted_scores

# 9️⃣ Output Results
print(df_scores)
df_scores.to_csv("Final_Supplier_Scoring_With_Criteria.csv", index=True)

print("✅ Predictions completed and saved successfully.")


Failed to crawl https://en.pouchen.com/about_pouchen.php: HTTPSConnectionPool(host='en.pouchen.com', port=443): Max retries exceeded with url: /about_pouchen.php (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x000001B41FCDE128>: Failed to establish a new connection: [Errno 11001] getaddrinfo failed'))


ValueError: in user code:

    File "C:\Users\srahman3\AppData\Local\anaconda3\envs\tf_clean\lib\site-packages\keras\engine\training.py", line 2041, in predict_function  *
        return step_function(self, iterator)
    File "C:\Users\srahman3\AppData\Local\anaconda3\envs\tf_clean\lib\site-packages\keras\engine\training.py", line 2027, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "C:\Users\srahman3\AppData\Local\anaconda3\envs\tf_clean\lib\site-packages\keras\engine\training.py", line 2015, in run_step  **
        outputs = model.predict_step(data)
    File "C:\Users\srahman3\AppData\Local\anaconda3\envs\tf_clean\lib\site-packages\keras\engine\training.py", line 1983, in predict_step
        return self(x, training=False)
    File "C:\Users\srahman3\AppData\Local\anaconda3\envs\tf_clean\lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "C:\

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv(
    r"C:\Users\srahman3\OneDrive - The University of Texas at El Paso\Desktop\Summer 25\Supplier Selection\V5 Final\IEEE TEM\Final three TEM\Suppier_Disruption_LogTable_Coded.csv"
)

df.head()

,date,supplier,kg_score,Em_Score,twitter_alert,weather_risk,economic_risk,geo_risk,tech_risk,downtime_hours,high_disruption_p
0,1/1/2025,JHH,0.442711,0.552985,1,0,1,0,0,5.1,0.866602
1,1/2/2025,JHH,0.599101,0.509835,0,1,0,0,0,7.2,0.847657
2,1/3/2025,JHH,0.372080,0.532829,0,0,0,0,0,1.0,0.746924
3,1/4/2025,JHH,1.167822,0.571556,0,0,0,0,0,1.8,0.960245
4,1/5/2025,JHH,0.071730,0.591891,1,1,0,0,0,7.7,0.752698


In [5]:
# Binary target using threshold 0.6
df["high_disruption_label"] = (df["high_disruption_p"] >= 0.6).astype(int)

# Feature set (edit if needed)
X = df[[
    "kg_score",
    "Em_Score",
    "twitter_alert",
    "weather_risk",
    "economic_risk",
    "geo_risk",
    "tech_risk",
    "downtime_hours",
   
]]

y = df["high_disruption_label"]

In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [11]:
models = {
    "Logistic Regression": LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        C=1.0,
        random_state=42
    ),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    ),
    
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.1,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        random_state=65,
        eval_metric="logloss",
        use_label_encoder=False
    )
}

In [13]:
results = []

for name, model in models.items():
    
    acc = cross_val_score(model, X, y, cv=skf, scoring="accuracy")
    f1 = cross_val_score(model, X, y, cv=skf, scoring="f1")
    roc = cross_val_score(model, X, y, cv=skf, scoring="roc_auc")
    
    results.append({
        "Model": name,
        "Accuracy Mean": np.mean(acc),
        "Accuracy Std": np.std(acc),
        "F1 Mean": np.mean(f1),
        "F1 Std": np.std(f1),
        "ROC-AUC Mean": np.mean(roc),
        "ROC-AUC Std": np.std(roc)
    })

cv_results = pd.DataFrame(results).sort_values("ROC-AUC Mean", ascending=False)

print("\n📊 5-Fold Cross Validation Results:\n")
print(cv_results)

C:\Users\srahman3\AppData\Local\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [09:27:27] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\srahman3\AppData\Local\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [09:27:28] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\srahman3\AppData\Local\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [09:27:28] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
C:\Users\srahman3\AppData\Local\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [09:27:28] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are no


📊 5-Fold Cross Validation Results:

                 Model  Accuracy Mean  Accuracy Std   F1 Mean    F1 Std  \
3              XGBoost       0.970000      0.017951  0.978496  0.012984   
1        Random Forest       0.961667      0.018708  0.972595  0.013182   
2    Gradient Boosting       0.968333      0.013333  0.977196  0.009530   
0  Logistic Regression       0.956667      0.020683  0.969450  0.014452   

   ROC-AUC Mean  ROC-AUC Std  
3      0.996386     0.002200  
1      0.996088     0.002887  
2      0.994691     0.003299  
0      0.989085     0.007955  


C:\Users\srahman3\AppData\Local\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [09:27:30] WARNING: D:\bld\xgboost-split_1737531313485\work\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
